# Visualizing v26 Shadow Log Generation — Cross-Section Analysis

This notebook generates a shadow log using v26 (MLE weighting), finds a mutated trace, and **profiles the exact Katz backoff + mutation decision**.

**Key concepts:**
- **Katz Backoff**: Tries N=6 → N=5 → … → N=1, falling back when data is sparse
- **P_unseen**: Good-Turing estimate — probability the next activity hasn't been seen in this context
- **v25+ Katz-Consistent Mutation**: When a mutation fires, the inserted activity is drawn from the **deepest lower-order context** that offers genuinely novel continuations, not from the uniform alphabet
- **Acceptance rate (v26)**: `gen_accept` — fraction of traces replayed perfectly by the model
- **MLE weighting (v26)**: Samples proportionally to raw frequencies (vs `'log'` damping)

In [31]:
import random
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11
})

import pm4py

# Load the Sepsis log
log = pm4py.read_xes("data/Sepsis Cases - Event Log_1_all/Sepsis Cases - Event Log.xes.gz")
log = pm4py.convert_to_event_log(log)
print(f"Log: {len(log)} traces")

# Pre-compute N-gram statistics (v26 default: max_n=6)
max_n = 6
ngram_outgoings = {n: defaultdict(Counter) for n in range(1, max_n + 1)}
starts, ends = Counter(), Counter()
global_counts = Counter()
alphabet = set()

for trace in log:
    seq = [e["concept:name"] for e in trace]
    starts[seq[0]] += 1
    ends[seq[-1]] += 1
    alphabet.update(seq)
    for act in seq:
        global_counts[act] += 1
    for i in range(len(seq) - 1):
        nxt = seq[i + 1]
        for n in range(1, max_n + 1):
            if i >= n - 1:
                state = tuple(seq[i - (n - 1) : i + 1])
                ngram_outgoings[n][state][nxt] += 1

alphabet = list(alphabet)
print(f"Alphabet: {len(alphabet)} activities")
for n in range(1, max_n + 1):
    print(f"  N={n}: {len(ngram_outgoings[n])} states")

Log: 1050 traces
Alphabet: 16 activities
  N=1: 16 states
  N=2: 99 states
  N=3: 358 states
  N=4: 833 states
  N=5: 1546 states
  N=6: 2442 states


## 2. Generate Shadow Log & Find a Mutation

We run v26's `generate_shadow_log` (with `successor_weighting='mle'`, the headline configuration) and locate a trace containing a mutation.

In [25]:
from HybridGen.algorithm.v26 import generate_shadow_log

# Generate shadow log using v26 (MLE weighting, the headline candidate)
random.seed(42)
shadow_log, mutation_flags, duplicates_kept, truncated, cap_used = generate_shadow_log(
    log, num_traces=500, safe_threshold=5, max_n=6, successor_weighting="mle"
)

trace_lens = [len(t) for t in shadow_log]
mutated = np.array(mutation_flags)

print("=" * 60)
print("  Shadow Log (v26, MLE weighting) — Summary")
print("=" * 60)
print(f"  Total traces:       {len(shadow_log)}")
print(f"  Mutated traces:     {sum(mutated)} ({sum(mutated)/len(shadow_log)*100:.1f}%)")
print(f"  Duplicates kept:    {duplicates_kept} (dedup cap never exhausted)")
print(f"  Truncated traces:   {truncated}")
print(f"  Max length used:    {cap_used}")

# Find the first mutated trace for our cross-section analysis
first_mut_idx = None
first_mut_trace = None
for i, (t, m) in enumerate(zip(shadow_log, mutation_flags)):
    if m:
        first_mut_idx = i
        first_mut_trace = [e["concept:name"] for e in t]
        break

print(f"\n  First mutated trace: #{first_mut_idx}")
print(f"  {' -> '.join(first_mut_trace)}")
print(f"  Length: {len(first_mut_trace)} events")

  Shadow Log (v26, MLE weighting) — Summary
  Total traces:       500
  Mutated traces:     269 (53.8%)
  Duplicates kept:    0 (dedup cap never exhausted)
  Truncated traces:   0
  Max length used:    370

  First mutated trace: #2
  ER Registration -> ER Triage -> ER Sepsis Triage -> LacticAcid -> CRP -> Leucocytes -> IV Antibiotics -> Admission NC -> Admission NC -> Leucocytes -> CRP -> Release D -> Return ER
  Length: 13 events


## 3. Mutation Cross-Section: Katz Backoff + v25 Proposal

We locate the mutation (Step 11), then drill into the Katz backoff resolution and the v25 Katz-consistent proposal chain that found `Release D` from a shallower context.

In [30]:
# =====================================================================
# Cross-Section: How the Mutation Decision Was Made
# =====================================================================

step = mut_step
context = tuple(first_mut_trace[:step]) if step > 0 else ()
current_local = first_mut_trace[step - 1] if step > 0 else None
actual_next = first_mut_trace[step]

print("=" * 60)
print("  Finding the Mutation Step")
print("=" * 60)
print()
for i, act in enumerate(first_mut_trace):
    marker = " ★ MUTATION" if i == mut_step else ""
    print(f"    [{i}] {act}{marker}")
print()

# ── Part 1: Katz Backoff Resolution ──────────────────────────────────────
print("=" * 60)
print("  Part 1: Katz Backoff — Resolve Context & P_unseen")
print("=" * 60)
print()
print(f"  Context: {' -> '.join(context)}")
print(f"  Question: Does this exact sequence have enough data to")
print(f"  predict the next activity?")
print()

resolved_n = None
resolved_out = None
p_unseen_confirmed = None
n_total_confirmed = None

print(f"  ┌─ Try N={max_n} → N=2 (safe_threshold = 5):")
for n in range(max_n, 1, -1):
    if len(context) >= n:
        state = context[-n:]
        out = ngram_outgoings[n].get(state, {})
        n_total = sum(out.values())
        n1 = sum(1 for c in out.values() if c == 1)

        if n_total == 0:
            print(f"  │  N={n}: no data → skip")
            continue

        p = n1 / n_total if n_total > 0 else 1.0

        if n_total < 5:
            print(f"  │  N={n}: n_total={n_total} < 5 (sparse) → backoff")
        elif p >= 1.0:
            print(f"  │  N={n}: n_total={n_total} ≥ 5, but ALL successors occur once")
            print(f"  │         (p_unseen = {n1}/{n_total} = 1.0) → backoff")
        else:
            print(f"  │  N={n}: n_total={n_total} ≥ 5, p_unseen = {n1}/{n_total} = {p:.4f}")
            print(f"  │  │    successors: {dict(out)}")
            print(f"  │  └─ ✓ RESOLVED here")
            resolved_n = n
            resolved_out = out
            p_unseen_confirmed = p
            n_total_confirmed = n_total
            break

if resolved_n is None:
    # Fallback to N=1
    out = ngram_outgoings[1].get((current_local,), {})
    n_total = sum(out.values())
    n1 = sum(1 for c in out.values() if c == 1)
    p = n1 / n_total if n_total > 0 else 1.0
    print(f"  │  N=1 (fallback): p_unseen = {n1}/{n_total} = {p:.4f}")
    print(f"  │  └─ ✓ RESOLVED at N=1")
    resolved_n = 1
    resolved_out = out
    p_unseen_confirmed = p
    n_total_confirmed = n_total

print(f"  │")
seen_at_resolved = set(resolved_out.keys())
print(f"  └─ Resolved at N={resolved_n}. Seen successors: {sorted(seen_at_resolved)}")
print(f"      P_unseen = {p_unseen_confirmed:.4f}  (probability next activity is novel)")
print()

# ── Part 2: Mutation Proposal ────────────────────────────────────────────
print("=" * 60)
print("  Part 2: Mutation Fires → Katz-Consistent Proposal")
print("=" * 60)
print()
print(f"  Roll < P_unseen → a mutation fires.")
print(f"  The inserted activity must be something the resolved")
print(f"  N={resolved_n} context has NEVER seen.")
print(f"  But it should be PLAUSIBLE (seen in SOME shorter context).")
print()
print(f"  ┌─ Scan N={min(len(context), max_n)} → N=1 (deepest-first),")
print(f"  │  keep only activities NOT in N={resolved_n}'s seen set.")
print(f"  │  STOP at the first level with any candidates.")
print()

found_source_n = None
for m in range(min(len(context), max_n), 0, -1):
    state = context[-m:] if m > 0 else ()
    out = ngram_outgoings[m].get(state, {})
    candidates = {a: c for a, c in out.items() if a not in seen_at_resolved}

    if m == resolved_n:
        print(f"  │  N={m} (the resolved level itself): successors = {dict(out) if out else '(empty)'}")
        print(f"  │     → These ARE 'seen_at_resolved' (exploit choices). Nothing new here.")
        print()
        continue

    if candidates:
        print(f"  │  >>> N={m}: FOUND candidates unseen at N={resolved_n}")
        print(f"  │      All successors: {dict(out)}")
        print(f"  │      Unseen at N={resolved_n}: {candidates}")
        print(f"  │      → STOP, ignore N={m-1}..N=1")
        if len(candidates) == 1:
            print(f"  │      → Only 1 candidate; random.choice picks it deterministically")
        else:
            print(f"  │      → Weighted random choice among {len(candidates)} candidates")
        print(f"  │  └─ Mutation inserts: '{actual_next}'")
        found_source_n = m
        break
    elif out:
        print(f"  │  N={m}: successors={dict(out)}")
        print(f"  │     → All are already in N={resolved_n}'s seen set. No new candidates.")
        print()
    else:
        print(f"  │  N={m}: (no successors in ngram)")
        print()

if not found_source_n:
    # Global frequency fallback
    cand = {a: c for a, c in global_counts.items() if a not in seen_at_resolved}
    if cand:
        print(f"  │  >>> Global frequency fallback:")
        print(f"  │      Candidates unseen in log: {cand}")
        print(f"  │  └─ Mutation inserts: '{actual_next}'")
    else:
        print(f"  │  >>> No unseen activities anywhere → exploit instead of mutate")

print()
print("  └─ Key insight:")
print(f"      '{actual_next}' was never seen after this exact N={resolved_n}")
print(f"      sequence. But it WAS seen after a SHORTER context (N={found_source_n}).")
print(f"      → Katz-consistent mutation: novel yet plausible.")

  Finding the Mutation Step

    [0] ER Registration
    [1] ER Triage
    [2] ER Sepsis Triage
    [3] LacticAcid
    [4] CRP
    [5] Leucocytes
    [6] IV Antibiotics
    [7] Admission NC
    [8] Admission NC
    [9] Leucocytes
    [10] CRP
    [11] Release D ★ MUTATION
    [12] Return ER

  Part 1: Katz Backoff — Resolve Context & P_unseen

  Context: ER Registration -> ER Triage -> ER Sepsis Triage -> LacticAcid -> CRP -> Leucocytes -> IV Antibiotics -> Admission NC -> Admission NC -> Leucocytes -> CRP
  Question: Does this exact sequence have enough data to
  predict the next activity?

  ┌─ Try N=6 → N=2 (safe_threshold = 5):
  │  N=6: n_total=5 ≥ 5, but ALL successors occur once
  │         (p_unseen = 5/5 = 1.0) → backoff
  │  N=5: n_total=35 ≥ 5, p_unseen = 1/35 = 0.0286
  │  │    successors: {'Leucocytes': 8, 'CRP': 10, 'Release A': 13, 'Admission NC': 3, 'LacticAcid': 1}
  │  └─ ✓ RESOLVED here
  │
  └─ Resolved at N=5. Seen successors: ['Admission NC', 'CRP', 'LacticAcid', 

## 4. Regular vs Mutated Traces

Mutated traces contain activities that the model's N-gram context had never seen as successors — injected by the Katz-consistent proposal from a shallower order.

In [27]:
print("=" * 70)
print("  REGULAR TRACES (no novel transitions)")
print("=" * 70)
regular_lens = [l for l, m in zip(trace_lens, mutation_flags) if not m]
mutated_lens = [l for l, m in zip(trace_lens, mutation_flags) if m]

seen = 0
for i, (t, m) in enumerate(zip(shadow_log, mutation_flags)):
    if not m and seen < 3:
        seq = [e["concept:name"] for e in t]
        print(f"  Trace #{i} ({len(seq)} events): {' -> '.join(seq)}")
        seen += 1

print()
print("=" * 70)
print("  MUTATED TRACES (contain novel transitions)")
print("=" * 70)
seen = 0
for i, (t, m) in enumerate(zip(shadow_log, mutation_flags)):
    if m and seen < 3:
        seq = [e["concept:name"] for e in t]
        print(f"  Trace #{i} ({len(seq)} events): {' -> '.join(seq)}")
        seen += 1

print()
print("=" * 70)
print("  Aggregate Comparison")
print("=" * 70)
print(f"  {'Metric':<35} {'Regular':<12} {'Mutated':<12}")
print(f"  {'─'*35} {'─'*12} {'─'*12}")
print(f"  {'Count':<35} {len(regular_lens):<12} {len(mutated_lens):<12}")
print(f"  {'Mean trace length':<35} {np.mean(regular_lens):<12.1f} {np.mean(mutated_lens):<12.1f}")
print(f"  {'Min length':<35} {min(regular_lens):<12} {min(mutated_lens):<12}")
print(f"  {'Max length':<35} {max(regular_lens):<12} {max(mutated_lens):<12}")

  REGULAR TRACES (no novel transitions)
  Trace #0 (12 events): ER Registration -> ER Triage -> Leucocytes -> CRP -> LacticAcid -> ER Sepsis Triage -> IV Antibiotics -> Admission NC -> IV Liquid -> CRP -> Leucocytes -> Release A
  Trace #1 (12 events): ER Registration -> ER Triage -> ER Sepsis Triage -> LacticAcid -> Leucocytes -> CRP -> Admission NC -> Admission NC -> IV Antibiotics -> Admission NC -> IV Liquid -> Release B
  Trace #3 (12 events): ER Registration -> ER Triage -> ER Sepsis Triage -> Leucocytes -> CRP -> LacticAcid -> IV Liquid -> IV Antibiotics -> Admission IC -> Leucocytes -> CRP -> LacticAcid

  MUTATED TRACES (contain novel transitions)
  Trace #2 (13 events): ER Registration -> ER Triage -> ER Sepsis Triage -> LacticAcid -> CRP -> Leucocytes -> IV Antibiotics -> Admission NC -> Admission NC -> Leucocytes -> CRP -> Release D -> Return ER
  Trace #6 (12 events): ER Registration -> ER Triage -> ER Sepsis Triage -> CRP -> LacticAcid -> Leucocytes -> IV Liquid -> IV Ant

## Key Takeaways for Presentation

| Concept | Description |
|---|---|
| **Gen_shadow pipeline** | N-gram stats → shadow log generation → token replay on model |
| **Katz Backoff** | N=6 → … → N=1: falls back when states are too sparse (total < 5) or P_unseen = 1.0 |
| **P_unseen** | Good-Turing estimate: N₁/N_total — probability the next event hasn't been seen after this context |
| **Mutation trigger** | `random() < P_unseen` → insert a novel activity |
| **v24- mutation** | Picks uniformly from the full alphabet (random noise) |
| **v25+ mutation** | **Katz-consistent:** scans lower-order contexts for activities unseen at resolved order → "plausible in a shorter context, novel in this exact one" |
| **MLE weighting (v26)** | `successor_weighting='mle'` samples proportionally to raw frequencies — **headline candidate** |
| **Acceptance rate (v26)** | `gen_accept` — fraction of traces replayed perfectly (vs partial-credit mean fitness) |
| **Probe integrity** | `duplicates_kept` (dedup cap exhaustion), `truncated_traces` (length cap hits) — both 0 in practice |
| **Mutation rate** | ~54% on Sepsis at N=6 — shows why v25's Katz-consistent proposal matters for sparse logs |